## An example of using Jupyter for Documenting and Automating BDD Style Tests

This is a sample document which contains both software feature requirements and its corresponding manual and automated tests. This is an executable document, which can be shared between Business Analyst, Developer, (manual and/or automation) Testers and other stakeholders.

Aggregating all these information in a single executable file can help maintain the synchronization between the requirements, manual tests and automated tests. It helps outdated requirements, outdated manual test steps or outdated automation test steps to be identified easily.

Because the document is written in Jupyter (IPython) notebook, tests can easily be arranged in BDD (Gherkin) style without requiring any third party BDD framework.

The tests can be executed one cell at a time, or all in one go. These tests [can be discovered and run using Pytest](https://github.com/ldiary/marigoso/blob/master/notebooks/using_pytest_to_execute_bdd_style_tests_written_in_jupyter_ipython_notebooks.ipynb), which means, these tests can also be imported into Continuous Integration test environment such as Jenkins.

## Requirements Summary

<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="text-align: center; padding: 8px;">Scenario</th>
      <th style="text-align: left; padding: 8px;">Can Not Post Comment as Anonymous</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 8px;"><strong>Given</strong></td>
      <td style="padding: 8px;"><a href="#Given-I-am-a-Blogger-anonymous-user.">I am a Blogger anonymous user.</a></td>
    </tr>
    <tr>
      <td style="padding: 8px;"><strong>And</strong></td>
      <td style="padding: 8px;"><a href="#And-I-am-not-logged-in-to-any-google-account.">I am not logged in to any google account.</a></td>
    </tr>
    <tr>
      <td style="padding: 8px;"><strong>When</strong></td>
      <td style="padding: 8px;"><a href="#When-I-post-a-comment-to-a-blog-post.">I post a comment to a blog post.</a></td>
    </tr>
    <tr>
      <td style="padding: 8px;"><strong>Then</strong></td>
      <td style="padding: 8px;"><a href="#Then-the-comment-input-must-be-successful.">the comment input must be successful.</a></td>
    </tr>
    <tr>
      <td style="padding: 8px;"><strong>But</strong></td>
      <td style="padding: 8px;"><a href="#But-I-must-be-prompted-to-login-first-before-post-can-be-completed.">I must be prompted to login first before post can be completed.</a></td>
    </tr>
  </tbody>
</table>

## Manual and Automated Test Steps

In [1]:
from pytest_testbook import pw_start, pw_execute, pw_stop
pw_start(False)

Step 01: Navigating to Blogger...
Landed on: Blogger.com - Create a unique and beautiful blog easily.
Expected Results verified successfully.
Verification successful: Full header detected using unique IDs.
Typing comment...
Comment drafted successfully.
Waiting for PUBLISH button inside iframe...
PUBLISH button clicked successfully.
Verifying comment visibility...


### Given I am a Blogger anonymous user.
<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="text-align: center; padding: 8px;">Step</th>
      <th style="text-align: left; padding: 8px;">Actions</th>
      <th style="text-align: left; padding: 8px;">Expected Results</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: center; padding: 8px;">01</td>
      <td style="padding: 8px;">Launch a browser and navigate to Blogger website.</td>
      <td style="padding: 8px;">The loaded page should contain a header asking you to sign in to Blogger.</td>
    </tr>
  </tbody>
</table>

In [2]:
def anonymous_blogger_user(state):
    # 1. Retrieve the persistent page from the background worker
    page = state['page']
    
    print("Step 01: Navigating to Blogger...")
    page.goto('https://www.blogger.com')
    
    # Wait for the page to fully load
    page.wait_for_load_state("networkidle")
    
    title = page.title()
    print(f"Landed on: {title}")
    
    # 2. Expected Results Assertion
    assert 'Blogger' in title, f"Expected 'Blogger' in title, but got '{title}'"
    
    # Look for the 'Sign in' text to confirm anonymous status
    # Note: Text matching is case-sensitive by default, so we convert to lower case for a safer check
    page_text = page.locator("body").inner_text().lower()
    assert 'sign in' in page_text, "Expected the page to ask the user to sign in."
    
    print("Expected Results verified successfully.")

# Execute the step in the background thread
pw_execute(anonymous_blogger_user)

### And I am not logged in to any google account.
<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="text-align: center; padding: 8px;">Step</th>
      <th style="text-align: left; padding: 8px;">Actions</th>
      <th style="text-align: left; padding: 8px;">Expected Results</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: center; padding: 8px;">02</td>
      <td style="padding: 8px;">Navigate to any other Google services you are subscribed to, e.g Gmail.</td>
      <td style="padding: 8px;">The loaded page should contain a header asking you to sign in to that Google service.</td>
    </tr>
  </tbody>
</table>

In [3]:
def check_gmail_logged_out(state):
    page = state['page']
    page.goto("https://mail.google.com/")
    page.wait_for_load_state("networkidle")
    
    # 1. Target the H1 using its ID for maximum stability
    title_header = page.locator("#headingText")
    
    # 2. Target the div using its unique ID discovered in the error log
    subtext_div = page.locator("#headingSubtext")
    
    # Wait for both to be visible
    title_header.wait_for(state="visible", timeout=10000)
    subtext_div.wait_for(state="visible", timeout=10000)
    
    # Assertions
    assert title_header.is_visible(), "Title element not found."
    assert subtext_div.is_visible(), "Subtext element not found."
    
    # Optional: Verify the text content inside those specific elements
    assert "Sign in" in title_header.inner_text(), "Title text mismatch."
    assert "to continue to Gmail" in subtext_div.inner_text(), "Subtext text mismatch."
    
    print("Verification successful: Full header detected using unique IDs.")

pw_execute(check_gmail_logged_out)

### When I post a comment to a blog post.
<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="text-align: center; padding: 8px;">Step</th>
      <th style="text-align: left; padding: 8px;">Actions</th>
      <th style="text-align: left; padding: 8px;">Expected Results</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: center; padding: 8px;">03</td>
      <td style="padding: 8px;">Navigate to a particular post in Blogger.</td>
      <td style="padding: 8px;">Page must load successfully.</td>
    </tr>
    <tr>
      <td style="text-align: center; padding: 8px;">04</td>
      <td style="padding: 8px;">If there is a Cookie Notice from Google, dismiss it.</td>
      <td style="padding: 8px;">Cookie notice must be dismissed successfully.</td>
    </tr>
    <tr>
      <td style="text-align: center; padding: 8px;">05</td>
      <td style="padding: 8px;">Provide the following input:<br><b>Comment body</b><br><b>Comment as</b></td>
      <td style="padding: 8px;">Input must be successfull.<br><i>An example of Selenium automation in Python.</i><br><i>Google Account</i></td>
    </tr>
  </tbody>
</table>

In [4]:
def post_blog_comment(state):
    # Retrieve the persistent page
    page = state['page']
    page.goto("http://pytestuk.blogspot.co.uk/2015/11/testing.html")

    # 1. Target the iframe using the unique class
    # We use .first to handle the multiple iframes Blogger loads
    frame = page.frame_locator(".comments-replybox").first

    # 2. Target the textarea using the accessible label (not the ID)
    # This automatically ignores hidden/duplicate elements
    comment_box = frame.get_by_label("Enter comment")

    # 3. Wait for the element to be ready
    comment_box.wait_for(state="visible", timeout=15000)

    # 4. Fill the comment
    print("Typing comment...")
    comment_box.fill("An example of Selenium automation in Python.")

    print("Comment drafted successfully.")

# Send to the background thread
pw_execute(post_blog_comment)

### Then the comment input must be successful. 
<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="text-align: center; padding: 8px;">Step</th>
      <th style="text-align: left; padding: 8px;">Actions</th>
      <th style="text-align: left; padding: 8px;">Expected Results</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: center; padding: 8px;">06</td>
      <td style="padding: 8px;">Press the "Publish" button at the buttom of the page.</td>
      <td style="padding: 8px;">The page must be submitted without errors.</td>
    </tr>
  </tbody>
</table>

In [5]:
def click_publish_button(state):
    page = state['page']
    
    # 1. Locate the iframe specifically by its class
    frame_loc = page.frame_locator(".comments-replybox").first
    
    # 2. Define the button locator inside the frame
    # We use 'get_by_role' with 'PUBLISH' (exact casing as your screenshot)
    # This ignores whether it's a <div>, <span>, or <button>
    publish_btn = frame_loc.get_by_role("button", name="PUBLISH")
    
    # 3. Wait for it specifically
    print("Waiting for PUBLISH button inside iframe...")
    publish_btn.wait_for(state="visible", timeout=15000)
    
    # 4. Click
    publish_btn.click()
    print("PUBLISH button clicked successfully.")

# Send to the background thread
pw_execute(click_publish_button)

### But I must be prompted to login first before post can be completed. 
<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="text-align: center; padding: 8px;">Step</th>
      <th style="text-align: left; padding: 8px;">Actions</th>
      <th style="text-align: left; padding: 8px;">Expected Results</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: center; padding: 8px;">07</td>
      <td style="padding: 8px;">Observe the landing page after submitting the "Publish" button.</td>
      <td style="padding: 8px;">The page must ask you to login to Blogger.</td>
    </tr>
  </tbody>
</table>

In [6]:
import time

def verify_publish_success(state):
    page = state['page']
    
    # 1. Define the comment we just posted
    expected_comment = "An example of Selenium automation in Python."
    
    # 2. Find the comment on the page
    # Using .first handles cases where you might have posted multiple times
    comment_locator = page.get_by_text(expected_comment).first
    
    # 3. Wait for it to appear (this handles the async update after the click)
    print("Verifying comment visibility...")
    comment_locator.wait_for(state="visible", timeout=15000)
    
    # 4. Assert it is actually visible
    assert comment_locator.is_visible(), "Failure: The published comment is not visible on the page."
    
    

# Run the verification
pw_execute(verify_publish_success)

TimeoutError: Locator.wait_for: Timeout 15000ms exceeded.
Call log:
  - waiting for get_by_text("An example of Selenium automation in Python.").first to be visible


In [7]:
pw_stop()